# Generate sentence-based chunk embeddings

Split each legal article at sentence boundaries, group sentences up to the size limit, and generate SEA-LION and Granite R2 embeddings. This is an experimental alternative to the IBM-style recursive chunker.

In [1]:
import gc
import json
import logging
import re
from pathlib import Path

import numpy as np

logging.getLogger('torch.distributed.elastic.multiprocessing.redirects').setLevel(logging.ERROR)
try:
    from transformers.utils import logging as transformers_logging
    transformers_logging.disable_progress_bar()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data-pipeline').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
INPUT_PATH = DATA_DIR / 'processed' / 'search' / 'articles_normalized.jsonl'
CHUNK_PATH = DATA_DIR / 'processed' / 'search' / 'article_sentence_chunks.jsonl'
OUTPUT_DIR = DATA_DIR / 'processed' / 'embeddings'
MAX_CHARS, OVERLAP_CHARS = 1200, 200
SENTENCE_END_RE = re.compile(r'(?<=[។៕!?！？])')
MODELS = {
    'sealion': {'model': 'aisingapore/SEA-LION-E5-Embedding-600M', 'batch_size': 8, 'output': 'sentence_chunks_sealion_e5_600m'},
    'granite-r2': {'model': 'ibm-granite/granite-embedding-311m-multilingual-r2', 'batch_size': 2, 'output': 'sentence_chunks_granite_311m_r2'},
}

def split_sentences(text):
    text = text.replace('\r\n', '\n').strip()
    sentences = []
    for line in text.splitlines():
        line = re.sub(r'[ \t]+', ' ', line).strip()
        if not line:
            continue
        parts = [part.strip() for part in SENTENCE_END_RE.split(line) if part.strip()]
        sentences.extend(parts)
    return sentences

def split_long_sentence(sentence):
    if len(sentence) <= MAX_CHARS:
        return [sentence]
    words = sentence.split(' ')
    chunks, current = [], ''
    for word in words:
        candidate = f'{current} {word}' if current else word
        if len(candidate) <= MAX_CHARS:
            current = candidate
        else:
            if current: chunks.append(current)
            current = word
    if current: chunks.append(current)
    return chunks

def split_article(text):
    sentences = [part for sentence in split_sentences(text) for part in split_long_sentence(sentence)]
    chunks, current = [], ''
    for sentence in sentences:
        candidate = f'{current} {sentence}' if current else sentence
        if len(candidate) <= MAX_CHARS:
            current = candidate
            continue
        if current:
            chunks.append(current.strip())
        available = max(0, MAX_CHARS - len(sentence) - 1)
        overlap = chunks[-1][-min(OVERLAP_CHARS, available):] if chunks and available else ''
        current = f'{overlap}\n{sentence}'.strip() if overlap else sentence
    if current:
        chunks.append(current.strip())
    return chunks

def create_chunks():
    articles = [json.loads(line) for line in INPUT_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    output = []
    for article in articles:
        chunks = split_article(article.get('search_text', article.get('text', '')))
        for index, text in enumerate(chunks, start=1):
            output.append({'chunk_id': f"{article['id']}-sentence-chunk-{index}", 'parent_id': article['id'], 'article_id': article['article_id'], 'document_id': article['document_id'], 'article_number': article['article_number'], 'chunk_index': index, 'chunk_count': len(chunks), 'text': text, 'search_text': text, 'document_title': article.get('document_title'), 'document_number': article.get('document_number'), 'document_type': article.get('document_type'), 'source_url': article.get('source_url'), 'pdf_url': article.get('pdf_url'), 'chapter_number': article.get('chapter_number'), 'chapter_title': article.get('chapter_title'), 'section_number': article.get('section_number'), 'section_title': article.get('section_title')})
    CHUNK_PATH.parent.mkdir(parents=True, exist_ok=True)
    CHUNK_PATH.write_text(''.join(json.dumps(record, ensure_ascii=False) + '\n' for record in output), encoding='utf-8')
    return output

def encode(records, model_name, batch_size):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name)
    vectors = model.encode([record['search_text'] for record in records], batch_size=batch_size, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
    del model
    gc.collect()
    return np.asarray(vectors, dtype='float32')

def write_outputs(records, vectors, config):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    prefix = config['output']
    np.save(OUTPUT_DIR / f'{prefix}.npy', vectors)
    metadata_path = OUTPUT_DIR / f'{prefix}_metadata.jsonl'
    metadata_path.write_text(''.join(json.dumps({key: value for key, value in record.items() if key != 'search_text'}, ensure_ascii=False) + '\n' for record in records), encoding='utf-8')
    manifest = {'model': config['model'], 'provider': 'sentence-transformer', 'input_path': str(CHUNK_PATH), 'chunk_count': len(records), 'embedding_dimension': int(vectors.shape[1]), 'normalized_embeddings': True}
    (OUTPUT_DIR / f'{prefix}_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

print('Sentence chunk input:', CHUNK_PATH)

Sentence chunk input: /Users/nak/CADT/projects/legal-rag/data-pipeline/data/processed/search/article_sentence_chunks.jsonl


In [2]:
records = create_chunks()
results = {}
for model_key, config in MODELS.items():
    vectors = encode(records, config['model'], config['batch_size'])
    write_outputs(records, vectors, config)
    results[model_key] = {'chunks': len(records), 'shape': list(vectors.shape)}
    del vectors
    gc.collect()
print(json.dumps(results, ensure_ascii=False, indent=2))

{
  "sealion": {
    "chunks": 3071,
    "shape": [
      3071,
      1024
    ]
  },
  "granite-r2": {
    "chunks": 3071,
    "shape": [
      3071,
      768
    ]
  }
}
